In [1]:
from pathlib import Path
import sys
import numpy as np

import torch
import pandas as pd
import tifffile
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import WeightedRandomSampler

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models import UNet

In [2]:
from src.data import MRIDataset, build_patient_params, build_weighted_sampler, build_loaders
from src.models import UNet, build_model
from src.training import bce_dice_loss, train_one_epoch, save_checkpoint, run_experiments
from src.evaluation import (
    validate_one_epoch,
    evaluate_patients_3d,
    summarize_evaluation,
    summarize_runs
)

In [3]:
pairs_df = pd.read_csv(project_root / "data" / "metadata" / "pairs.csv")
splits_df = pd.read_csv(project_root / "data" / "splits" / "splits.csv")
pairs_with_split = pairs_df.merge(
    splits_df, on="patient_id", how="left", validate="many_to_one"
)

val_df = pairs_with_split.loc[pairs_with_split["split"] == "val"].copy()
train_df = pairs_with_split.loc[pairs_with_split["split"] == "train"].copy()

print(f"Train: {len(train_df)} przekrojów, {train_df['patient_id'].nunique()} pacjentów.")
print(f"Val: {len(val_df)} przekrojów, {val_df['patient_id'].nunique()} pacjentów.")

Train: 2714 przekrojów, 77 pacjentów.
Val: 615 przekrojów, 16 pacjentów.


Hipoteza: przycięcie i przeskalowanie według percentyli 10/99 przed z-score poprawi średni walidacyjny Dice 3D po pacjentach. Porównujemy próbkowanie + z-score z próbkowaniem + percentyle + z-score. Oba warianty: stały split z splits.csv, seedy 42/43/44, 40 epok, checkpoint według najwyższego Dice 3D przed filtrem. Raportujemy pięć metryk na CPU oraz średnie i odchylenia między seedami (ddof=1). Test pozostaje odłożony.

In [10]:
config = {
    "experiment_name": "unet_sampling_percentile_zscore_dice3d",
    "seeds": [42, 43, 44],
    "model": "unet",
    "optimizer": "adam",
    "loss": "bce_dice",
    "learning_rate": 3e-4,
    "batch_size": 4,
    "num_epochs": 40,
    "preprocessing": "patient_percentile_zscore",
    "sampling": "patient_area",
    "threshold": 0.5,
    "selection_metric": "dice_3d",
    "save_last": True,
    "percentiles": [10, 99],
    "run_training": False,
}

In [5]:
match config["preprocessing"]:
    case "patient_channel_zscore":
        train_params = build_patient_params(train_df, project_root)
        val_params = build_patient_params(val_df, project_root)
    case "scale_255":
        train_params = None
        val_params = None
    case "patient_percentile_zscore":
        train_params = build_patient_params(
            train_df,
            project_root,
            percentiles=config["percentiles"],
        )
        val_params = build_patient_params(
            val_df,
            project_root,
            percentiles=config["percentiles"],
        )
    case _:
        raise ValueError("nieznany preprocessing")        

In [6]:
train_dataset = MRIDataset(
    train_df, project_root, patient_params=train_params
)

val_dataset = MRIDataset(
    val_df, project_root, patient_params=val_params
)

print(f"Długości: {len(train_dataset)} i {len(val_dataset)}")
print(f"obraz: {train_dataset[0][0].shape}")
print(f"maska: {train_dataset[0][1].shape}")

Długości: 2714 i 615
obraz: torch.Size([3, 256, 256])
maska: torch.Size([1, 256, 256])


In [7]:
from datetime import timedelta
from time import perf_counter

started_at = perf_counter()
try:
    results_df = run_experiments(
        config, project_root, train_dataset, val_dataset
    )
finally:
    elapsed_seconds = perf_counter() - started_at
    print(f"\nCzas wykonania: {timedelta(seconds=round(elapsed_seconds))}")

results_df


Seed 42: train 679 paczki, val 154 paczki
run name:  unet_sampling_percentile_zscore_dice3d_seed42

epoka:  1
train loss:  0.47430965699689215
val metrics:  {'loss': 0.8588352757740796, 'dice': 0.6571966922107508, 'dice_annotated': 0.5134574825149345, 'dice_empty': 0.7467018469656992, 'dice_3d': 0.47858702340463344, 'dice_3d_filtered': 0.5100678667308102}

epoka:  2
train loss:  0.35809061260634517
val metrics:  {'loss': 0.8724976689350314, 'dice': 0.6685396132672705, 'dice_annotated': 0.5133553468701193, 'dice_empty': 0.7651715039577837, 'dice_3d': 0.4614292262452247, 'dice_3d_filtered': 0.4907339636673781}

epoka:  3
train loss:  0.3579357235349973
val metrics:  {'loss': 0.8421606117147741, 'dice': 0.39441007594994415, 'dice_annotated': 0.5998398168170351, 'dice_empty': 0.26649076517150394, 'dice_3d': 0.41232336236111655, 'dice_3d_filtered': 0.4784144260916612}

epoka:  4
train loss:  0.3112024133071007
val metrics:  {'loss': 0.8079046152229231, 'dice': 0.34376847143095685, 'dice_an

,run_name,seed,epoch,dice,dice_annotated,dice_empty,dice_3d,dice_3d_filtered
0,unet_sampling_percentile_zscore_dice3d_seed42,42,33,0.681487,0.733536,0.649077,0.708114,0.752500
1,unet_sampling_percentile_zscore_dice3d_seed43,43,40,0.678029,0.749950,0.633245,0.689218,0.773028
2,unet_sampling_percentile_zscore_dice3d_seed44,44,40,0.693307,0.730439,0.670185,0.690273,0.713652


In [8]:
series_summary = summarize_runs(results_df)
series_summary

,mean,std
metric,,
dice,0.684274,0.008011
dice_annotated,0.737975,0.010485
dice_empty,0.650836,0.018532
dice_3d,0.695868,0.010618
dice_3d_filtered,0.746393,0.030155


## Historyczna jednoepokowa próba integracyjna

Poniższa komórka odczytuje unet_modular_smoke_seed42, czyli wcześniejszą
próbę działania modułów po jednej epoce. Nie jest to podsumowanie pełnej
serii percentylowej. Zakończone porównanie obu serii po 40 epok znajduje
się w kolejnej sekcji i korzysta ze wszystkich seedów 42/43/44.

In [9]:
saved_results_df = pd.read_csv(
    project_root / "outputs" / "unet_modular_smoke_seed42_val_metrics.csv"
)

summarize_runs(saved_results_df)

,mean,std
metric,,
dice,0.628485,NaN
dice_annotated,0.548805,NaN
dice_empty,0.678100,NaN
dice_3d,0.479485,NaN
dice_3d_filtered,0.489090,NaN


## Zakończone porównanie: z-score i percentyle 10/99 + z-score

Obie serie korzystają z tego samego podziału pacjentów w splits.csv
(train 77/2714, val 16/615), seedów treningowych 42/43/44 i 40 epok.
Użyto U-Neta, Adama z lr=3e-4, paczek po 4, BCE + średniego soft Dice
po przekrojach (wagi 1/1, eps=1e-5), próbkowania patient_area i braku augmentacji.
Checkpoint wybierano według najwyższego walidacyjnego Dice 3D przed
filtrowaniem; przy remisie zachowywano wcześniejszą epokę.
Próg to sigmoid(logits) > 0.5, a filtr zachowuje największą składową 3D
z sąsiedztwem 6. Raporty poniżej pochodzą z oceny checkpointów na CPU.

Jedyna zmiana metody to dodanie globalnych percentyli 10/99 przed z-score.
Progi wyznaczamy z całego wolumenu pacjenta łącznie z kanałami i tłem;
średnie i odchylenia po transformacji liczymy osobno dla kanałów w float64.

Komórki porównania odczytują sześć istniejących raportów CSV z outputs.
Nie uruchamiają treningu. Outputs jest lokalny i ignorowany przez Git;
zapisane wydruki zachowują tabelę także bez dostępu do tych plików.
Wyniki 07–10, wybierane według dice_annotated, stanowią oddzielną serię.

In [ ]:
comparison_seeds = [42, 43, 44]
comparison_run_names = {
    "Z-score": "unet_sampling_zscore_dice3d",
    "Percentyle + z-score": "unet_sampling_percentile_zscore_dice3d",
}
comparison_results = {}

for label, prefix in comparison_run_names.items():
    reports = [
        pd.read_csv(
            project_root / "outputs" / f"{prefix}_seed{seed}_val_metrics.csv"
        )
        for seed in comparison_seeds
    ]
    series_results = pd.concat(reports, ignore_index=True)
    expected_names = [f"{prefix}_seed{seed}" for seed in comparison_seeds]
    if (
        series_results["seed"].tolist() != comparison_seeds
        or series_results["run_name"].tolist() != expected_names
    ):
        raise ValueError(f"Niezgodne raporty serii: {prefix}")
    comparison_results[label] = series_results

comparison_summary = pd.concat(
    {
        label: summarize_runs(frame)
        for label, frame in comparison_results.items()
    },
    axis=1,
)
comparison_summary.round(4)

Z-score         Percentyle + z-score        
                    mean     std                 mean     std
metric                                                       
dice              0.6353  0.1347               0.6843  0.0080
dice_annotated    0.7163  0.0439               0.7380  0.0105
dice_empty        0.5849  0.1913               0.6508  0.0185
dice_3d           0.6779  0.0738               0.6959  0.0106
dice_3d_filtered  0.7239  0.0527               0.7464  0.0302

In [ ]:
control_runs = comparison_results["Z-score"].set_index("seed")
percentile_runs = comparison_results["Percentyle + z-score"].set_index("seed")

paired_comparison = pd.DataFrame({
    "epoka_zscore": control_runs["epoch"],
    "epoka_percentyle": percentile_runs["epoch"],
    "dice_3d_zscore": control_runs["dice_3d"],
    "dice_3d_percentyle": percentile_runs["dice_3d"],
    "roznica_percentyle_minus_zscore": (
        percentile_runs["dice_3d"] - control_runs["dice_3d"]
    ),
}).sort_index()
paired_comparison.round(4)

,epoka_zscore,epoka_percentyle,dice_3d_zscore,dice_3d_percentyle,roznica_percentyle_minus_zscore
seed,,,,,
42,36,33,0.7160,0.7081,-0.0079
43,30,40,0.7248,0.6892,-0.0356
44,37,40,0.5928,0.6903,0.0975


### Wniosek z zakończonej serii

Średni Dice 3D przed filtrem wzrósł z 0,6779 ± 0,0738 do
0,6959 ± 0,0106, czyli o 0,0180. Po filtrze średnia wzrosła
z 0,7239 ± 0,0527 do 0,7464 ± 0,0302, czyli o 0,0225.
Są to średnie i odchylenia standardowe z próby między trzema seedami
(ddof=1), obliczone z raportów CPU.

Wzrost nie wystąpił na każdym seedzie: zmiany surowego Dice 3D wyniosły
-0,0079 / -0,0356 / +0,0975 dla 42/43/44. Średnią podnosi głównie
seed 44, na którym poprawę odnotowano u 13 z 16 pacjentów.
Wybrane epoki kontroli to 36/30/37, a percentyli 33/40/40.

Wynik wspiera wykorzystanie percentyli jako kandydata do dalszych prób,
z zastrzeżeniem trzech seedów i jednego podziału walidacyjnego używanego
również do wyboru checkpointów. Mniejszy rozrzut wybranych wyników
nie dowodzi jeszcze ogólnej stabilizacji treningu.
Czas komórki run_experiments dla serii percentylowej wyniósł 1:08:22;
przygotowanie parametrów pacjentów odbywało się wcześniej, poza pomiarem.
Test pozostaje odłożony do zamrożenia protokołu.